In [1]:
import re
import math
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


veri = [
    ("rüzgar eser ağaçlar fısıldar sessiz akşam", 0),
    ("mavi gökyüzü altında sakin deniz dalgaları", 0),
    ("orman yolunda yürürken kuş sesleri duyulur", 0),
    ("sabah güneşi toprağı ısıtır çiy damlaları parlar", 0),
    ("dağ başında temiz hava ve derin sessizlik", 0),
    ("göl kenarında yavaş akan su ve sarı yapraklar", 0),
    ("yağmur yağar toprak kokusu yayılır her yere", 0),
    ("ay ışığında parlayan nehir ve serin gece", 0),
    ("yeşil vadide açan çiçekler ve hafif esinti", 0),
    ("kırsal patikada huzurlu yürüyüş ve güneş ışığı", 0),
    ("gece yarısı sokak lambaları ve rüzgarlı sahil", 0),
    ("bahar sabahı kuşlar uyanır doğa canlanır", 0),

    ("kalabalık cadde trafik ışıkları ve korna sesleri", 1),
    ("metro istasyonu sabah koşturmacası ve hızlı adımlar", 1),
    ("yüksek binalar neon ışıklar ve yoğun akşam trafiği", 1),
    ("otobüs durağında bekleyen yolcular ve şehir gürültüsü", 1),
    ("alışveriş merkezi kalabalığı ve hızlı yaşam temposu", 1),
    ("köprü trafiği siren sesleri ve hareketli sokaklar", 1),
    ("sabah erken saatte tren garı ve telaşlı kalabalık", 1),
    ("şehir merkezinde yoğun insan akışı ve ışıklar", 1),
    ("hızlı tren hattı gürültülü kavşak ve hareket", 1),
    ("akşam saatinde tıkanan yollar ve şehir telaşı", 1),
    ("iş çıkışında kalabalık sokaklar ve acele adımlar", 1),
    ("merkez meydanda hareketli kalabalık ve ışıklar", 1),
]

df = pd.DataFrame(veri, columns=["metin", "etiket"])
print("Toplam örnek sayısı:", len(df))
print("Sınıf dağılımı:\n", df["etiket"].value_counts())
df.head()

Toplam örnek sayısı: 24
Sınıf dağılımı:
 etiket
0    12
1    12
Name: count, dtype: int64


,metin,etiket
0,rüzgar eser ağaçlar fısıldar sessiz akşam,0
1,mavi gökyüzü altında sakin deniz dalgaları,0
2,orman yolunda yürürken kuş sesleri duyulur,0
3,sabah güneşi toprağı ısıtır çiy damlaları parlar,0
4,dağ başında temiz hava ve derin sessizlik,0


In [2]:

stop_words = {"ve", "ile", "her", "bu", "bir", "da", "de"}

def tokenize_ve_temizle(metin):

    metin = metin.lower()
    metin = re.sub(r"[^a-zçğıöşü0-9\s]", " ", metin)
    tokens = metin.split()
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    return tokens

df["tokens"] = df["metin"].apply(tokenize_ve_temizle)
for i in range(5):
    print(f"Örnek {i+1}: {df.loc[i, 'metin']} -> {df.loc[i, 'tokens']}")

Örnek 1: rüzgar eser ağaçlar fısıldar sessiz akşam -> ['rüzgar', 'eser', 'ağaçlar', 'fısıldar', 'sessiz', 'akşam']
Örnek 2: mavi gökyüzü altında sakin deniz dalgaları -> ['mavi', 'gökyüzü', 'altında', 'sakin', 'deniz', 'dalgaları']
Örnek 3: orman yolunda yürürken kuş sesleri duyulur -> ['orman', 'yolunda', 'yürürken', 'kuş', 'sesleri', 'duyulur']
Örnek 4: sabah güneşi toprağı ısıtır çiy damlaları parlar -> ['sabah', 'güneşi', 'toprağı', 'ısıtır', 'çiy', 'damlaları', 'parlar']
Örnek 5: dağ başında temiz hava ve derin sessizlik -> ['dağ', 'başında', 'temiz', 'hava', 'derin', 'sessizlik']


In [17]:

X_train_tokens, X_test_tokens, y_train, y_test = train_test_split(
    df["tokens"].tolist(),
    df["etiket"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["etiket"].tolist()
)

vocab = {"<UNK>": 0}
for doc in X_train_tokens:
    for token in doc:
        if token not in vocab:
            vocab[token] = len(vocab)

print("Eğitim sözlüğü boyutu (vocab size):", len(vocab))

def sayma_vektoru_olustur(doc, vocab):
    vec = np.zeros(len(vocab), dtype=float)
    for token in doc:
        idx = vocab.get(token, vocab["<UNK>"])
        vec[idx] += 1.0
    return vec

X_train_count = np.array([sayma_vektoru_olustur(doc, vocab) for doc in X_train_tokens])
X_test_count = np.array([sayma_vektoru_olustur(doc, vocab) for doc in X_test_tokens])

print("Eğitim sayma matrisi boyutu:", X_train_count.shape)
print("Test sayma matrisi boyutu:", X_test_count.shape)

Eğitim sözlüğü boyutu (vocab size): 104
Eğitim sayma matrisi boyutu: (19, 104)
Test sayma matrisi boyutu: (5, 104)


In [13]:

def tf_idf_hesapla(docs_tokens, vocab):
    N = len(docs_tokens)
    V = len(vocab)
    tf_mat = np.zeros((N, V), dtype=float)
    df_counts = np.zeros(V, dtype=float)

    for i, doc in enumerate(docs_tokens):
        uzunluk = len(doc)
        if uzunluk == 0:
            continue
        sayac = {}
        for token in doc:
            idx = vocab.get(token, vocab["<UNK>"])
            sayac[idx] = sayac.get(idx, 0) + 1
        for idx, adet in sayac.items():
            tf_mat[i, idx] = adet / uzunluk

    for j in range(V):

        doc_sayisi = 0
        for doc in docs_tokens:
            indeksler = {vocab.get(t, vocab["<UNK>"]) for t in doc}
            if j in indeksler:
                doc_sayisi += 1
        df_counts[j] = doc_sayisi


    idf_vec = np.log((N + 1.0) / (df_counts + 1.0)) + 1.0
    tfidf_mat = tf_mat * idf_vec
    return tfidf_mat, idf_vec

X_train_tfidf, idf_vec = tf_idf_hesapla(X_train_tokens, vocab)

N_test = len(X_test_tokens)
V = len(vocab)
tf_test = np.zeros((N_test, V), dtype=float)
for i, doc in enumerate(X_test_tokens):
    if len(doc) == 0:
        continue
    sayac = {}
    for token in doc:
        idx = vocab.get(token, vocab["<UNK>"])
        sayac[idx] = sayac.get(idx, 0) + 1
    for idx, adet in sayac.items():
        tf_test[i, idx] = adet / len(doc)
X_test_tfidf = tf_test * idf_vec

def oklit_uzakligi(v1, v2):
    return float(np.linalg.norm(v1 - v2))

def kosinus_benzerligi(v1, v2):
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return float(np.dot(v1, v2) / (norm1 * norm2))

print("Test[0] ile Train[0] Öklit Uzaklığı:", round(oklit_uzakligi(X_test_tfidf[0], X_train_tfidf[0]), 4))
print("Test[0] ile Train[0] Kosinüs Benzerliği:", round(kosinus_benzerligi(X_test_tfidf[0], X_train_tfidf[0]), 4))

Test[0] ile Train[0] Öklit Uzaklığı: 2.5785
Test[0] ile Train[0] Kosinüs Benzerliği: 0.0


In [14]:
class MarkovDilModeli:
    def __init__(self, vocab):
        self.vocab = vocab
        self.idx2token = {idx: tok for tok, idx in vocab.items()}
        self.V = len(vocab)
        self.pi = None
        self.A = None

    def egit(self, cumle_tokens_listesi):
        pi_sayac = np.zeros(self.V, dtype=float)
        A_sayac = np.zeros((self.V, self.V), dtype=float)
        N = len(cumle_tokens_listesi)

        for tokens in cumle_tokens_listesi:
            if not tokens:
                continue
            ilk_idx = self.vocab.get(tokens[0], self.vocab["<UNK>"])
            pi_sayac[ilk_idx] += 1.0

            for t in range(1, len(tokens)):
                onceki_idx = self.vocab.get(tokens[t - 1], self.vocab["<UNK>"])
                simdiki_idx = self.vocab.get(tokens[t], self.vocab["<UNK>"])
                A_sayac[onceki_idx, simdiki_idx] += 1.0


        self.pi = (pi_sayac + 1.0) / (N + self.V)
        self.A = np.zeros((self.V, self.V), dtype=float)
        for i in range(self.V):
            toplam_i = np.sum(A_sayac[i])
            self.A[i] = (A_sayac[i] + 1.0) / (toplam_i + self.V)

    def cumle_uret(self, uzunluk=6, seed=42):
        rng = np.random.default_rng(seed)

        mevcut_idx = int(rng.choice(self.V, p=self.pi))
        uretilen = [self.idx2token[mevcut_idx]]


        for _ in range(1, uzunluk):
            sonraki_idx = int(rng.choice(self.V, p=self.A[mevcut_idx]))
            uretilen.append(self.idx2token[sonraki_idx])
            mevcut_idx = sonraki_idx

        return " ".join(uretilen)

markov_model = MarkovDilModeli(vocab)
markov_model.egit(X_train_tokens)

print("π vektörü toplamı (1.0 olmalı):", round(float(np.sum(markov_model.pi)), 4))
print("A[0] satırı toplamı (1.0 olmalı):", round(float(np.sum(markov_model.A[0])), 4))
for s in range(3):
    print(f"Üretilen cümle {s+1}:", markov_model.cumle_uret(uzunluk=6, seed=10 + s))

π vektörü toplamı (1.0 olmalı): 1.0
A[0] satırı toplamı (1.0 olmalı): 1.0
Üretilen cümle 1: ışıklar yolunda koşturmacası başında vadide dağ
Üretilen cümle 2: kalabalık vadide dalgaları patikada dağ binalar
Üretilen cümle 3: duyulur ışıklar sessizlik derin bekleyen sesleri


In [16]:

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_tfidf, y_train)
y_pred_dt = dt_model.predict(X_test_tfidf)

print("=== Temel Karar Ağacı Test Sonuçları ===")
print("Accuracy :", round(accuracy_score(y_test, y_pred_dt), 4))
print("Precision:", round(precision_score(y_test, y_pred_dt, zero_division=0), 4))
print("Recall   :", round(recall_score(y_test, y_pred_dt, zero_division=0), 4))
print("F1 Score :", round(f1_score(y_test, y_pred_dt, zero_division=0), 4))


param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [2, 3, 5, None],
    "min_samples_split": [2, 3, 4]
}

grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="f1"
)
grid_search.fit(X_train_tfidf, y_train)

print("\n=== Grid Search + 5-Fold CV Sonuçları ===")
print("En iyi hiper-parametreler:", grid_search.best_params_)
print("En iyi CV F1 skoru:", round(grid_search.best_score_, 4))

en_iyi_model = grid_search.best_estimator_
y_pred_best = en_iyi_model.predict(X_test_tfidf)

print("\n=== En İyi Model Test Sonuçları ===")
print("Accuracy :", round(accuracy_score(y_test, y_pred_best), 4))
print("Precision:", round(precision_score(y_test, y_pred_best, zero_division=0), 4))
print("Recall   :", round(recall_score(y_test, y_pred_best, zero_division=0), 4))
print("F1 Score :", round(f1_score(y_test, y_pred_best, zero_division=0), 4))
print("\nSınıflandırma Raporu:\n", classification_report(y_test, y_pred_best, zero_division=0))

=== Temel Karar Ağacı Test Sonuçları ===
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0

=== Grid Search + 5-Fold CV Sonuçları ===
En iyi hiper-parametreler: {'criterion': 'gini', 'max_depth': 3, 'min_samples_split': 2}
En iyi CV F1 skoru: 0.7333

=== En İyi Model Test Sonuçları ===
Accuracy : 0.8
Precision: 1.0
Recall   : 0.5
F1 Score : 0.6667

Sınıflandırma Raporu:
               precision    recall  f1-score   support

           0       0.75      1.00      0.86         3
           1       1.00      0.50      0.67         2

    accuracy                           0.80         5
   macro avg       0.88      0.75      0.76         5
weighted avg       0.85      0.80      0.78         5

